# TL-Bot - char_classifier Training

**Before the very first run:**
- **Colab:** set runtime to GPU — *Runtime → Change runtime type → T4 GPU → Save*
- **Kaggle:** enable GPU (*Settings → Accelerator → GPU T4 x2*); add `RCLONE_TOKEN` secret (see below)
- **Lightning AI:** open a Studio with a T4 GPU before running this notebook
- **Local:** ensure `.venv` is active and a CUDA GPU is available (CPU works for smoke tests)

**Every session: run all cells top to bottom.**
- Cell 1 — training config: scripts and epoch count.
- Cell 2 — platform config: determines the platform, then mounts/installs/configures storage and clones the repo (Colab/Kaggle/Lightning), or confirms local paths.
- Cell 3 — starts or resumes training using cell 2's resolved config. Child-process output is piped back into the cell; without that Colab sends it to the server log and failures surface as a bare `CalledProcessError`.
- Cell 4 — final results once cell 3 completes: run summary, the metrics of the epoch saved as `best.pt`, and `curves.png`. Refuses to report on an unfinished run.

Checkpoints are saved after every epoch and persist across sessions on all platforms.

---
**One-time: zip and upload the dataset**
```powershell
.venv\Scripts\python.exe Models\remote_train.py --zip-dataset
# Colab/Kaggle: upload char-dataset.zip to My Drive/Colab Notebooks/TL-Bot/
# Lightning:    upload char-dataset.zip to /teamspace/studios/this_studio/TL-Bot/
# Local:        dataset is already present — no zip needed
```

**One-time: Kaggle rclone setup**
```powershell
# 1. Configure rclone (if not already done)
rclone config   # → New remote → name: gdrive → type: Google Drive → follow OAuth flow

# 2. Copy the token JSON to your clipboard
(Get-Content "$env:APPDATA\rclone\rclone.conf" | Select-String "^token = ").ToString().Replace("token = ", "") | Set-Clipboard

# 3. Add a Kaggle Secret: Add-ons → Secrets → Add new secret
#    Name: RCLONE_TOKEN   Value: paste clipboard (the {"access_token":...} JSON)
```

**Kaggle checkpoint sync:** `remote_train.py --sync-to` pushes checkpoints to Drive every 10 minutes during training and once on finish, so no manual sync cell is needed. To force a push after a crash:
`!rclone sync /kaggle/working/TL-Bot/checkpoints "gdrive:Colab Notebooks/TL-Bot/checkpoints/"`

---
## Cell 1 - Training Config
Training hyperparameters only — scripts, epoch count, scheduler, learning rate, resume behavior. No platform-specific settings here; platform detection and all platform-specific setup live in cell 2.

In [ ]:
# Scripts to train: "latin" | "kana" | "hangul" | "cjk" | "all"
# Single script -> checkpoints/<script>/   "all" -> checkpoints/
SCRIPTS = ["latin"]

# Epochs for this run.
# RESUME defaults to True -- checking progress.json before each session (cell 3
# prints it automatically) is how to decide whether continuing is worthwhile or
# hyperparameters need a change, not a hardcoded per-script flag. best.pt only
# ever updates on a genuine score improvement (see train.py), so it can't
# regress from a bad resume or a bad fresh attempt either way.
#
# Latin config history (2026-08-07/08): run 1 (LR 3e-4, unfreeze 4, GRID_MODE
# "all", 60 epochs) peaked val_acc 0.576 at epoch 9 then plateaued. run 2 (LR
# 1e-4, unfreeze 2) converged markedly slower with no proven benefit -- reverted.
# Current config drops GRID_MODE to "single" to remove grid-rotation
# augmentation variants, which rotate the focus glyph itself up to 270 degrees
# -- turning rotation-ambiguous Latin letters (b/q, d/p, n/u, 6/9, M/W) into
# each other under a fixed label; confirmed present but not dominant in a
# confused-pairs check.
EPOCHS = 24

SCHEDULER = "cosine"   # cosine (recommended) | cosine-warm | none
LR        = 3e-4       # head LR; backbone uses LR * 0.1
RESUME    = True        # keep True -- see note above

# Dataset variant to train against. Defaults to "char-dataset-ctx-small" --
# run 5, the string-rendered + target-glyph-cropped tiles (see
# render_chars_context.py) at count-parity with the original char-dataset
# (77,465 vs 77,799 images, tile_size=64) -- this is the comparison run
# against run 4 (0.4814 @ epoch 15, char-dataset, crop-scale 0.40-0.75).
# Set back to "char-dataset" to reproduce/continue runs 1-4 instead.
# Any name must already be zipped and uploaded to the platform's storage root:
#   python Models/remote_train.py --zip-dataset --dataset-name <name>
# A non-default name automatically gets its own checkpoints/<script>_<suffix>
# dir (mirrors remote_train.py's _make_ckpt_dir), so a comparison run can
# never land in and overwrite the default dataset's checkpoint.
DATASET_NAME = "char-dataset-ctx-small"


---
## Cell 2 - Platform Config
Determines `PLATFORM`, then a single `match`/`case` block per platform mounts/installs/configures storage, clones the repo, and sets `REPO_DIR`/`STORAGE_ROOT`/`storage_args` for cell 3. Nothing downstream re-branches on `PLATFORM`.

In [ ]:
import os
from pathlib import Path

PLATFORM = "local"

# KAGGLE_KERNEL_RUN_TYPE is set by Kaggle's infrastructure; checking it is
# more reliable than checking for /kaggle/input, which the kaggle Python
# package creates on other platforms too.
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    PLATFORM = "kaggle"
else:
    try:
        import google.colab  # noqa: F401
        PLATFORM = "colab"
    except ImportError:
        if os.path.isdir("/teamspace/studios/this_studio"):
            PLATFORM = "lightning"

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"


def _clone_or_pull(repo_dir):
    import subprocess
    os.makedirs(repo_dir, exist_ok=True)
    if os.path.isdir(f"{repo_dir}/.git"):
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)


match PLATFORM:
    case "colab":
        from google.colab import drive

        COLAB_ROOT = "/content/drive/MyDrive/Colab Notebooks/TL-Bot"

        drive.mount('/content/drive')
        REPO_DIR = "/content/Discord-TL_Bot"
        _clone_or_pull(REPO_DIR)

        STORAGE_ROOT = COLAB_ROOT
        storage_args = ["--storage-root", COLAB_ROOT]

    case "kaggle":
        from kaggle_secrets import UserSecretsClient
        import subprocess
        import zipfile as _zipfile

        KAGGLE_ROOT = "/kaggle/working/TL-Bot"

        # Install rclone if not already present
        _which = subprocess.run(["which", "rclone"], capture_output=True)
        if _which.returncode != 0:
            subprocess.run(["curl", "-fsSL", "https://rclone.org/install.sh",
                            "-o", "/tmp/rclone_install.sh"], check=True)
            subprocess.run(["sudo", "bash", "/tmp/rclone_install.sh"], check=True)

        # Configure gdrive via env vars -- no config file needed.
        # RCLONE_TOKEN = the {"access_token":...} JSON from the "token = " line of rclone.conf.
        os.environ["RCLONE_CONFIG_GDRIVE_TYPE"]  = "drive"
        os.environ["RCLONE_CONFIG_GDRIVE_SCOPE"] = "drive"
        os.environ["RCLONE_CONFIG_GDRIVE_TOKEN"] = UserSecretsClient().get_secret("RCLONE_TOKEN").strip()
        r = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
        if "gdrive:" not in r.stdout:
            raise RuntimeError("gdrive remote not found - check RCLONE_TOKEN secret.")

        # Sync checkpoints from Drive (empty on first run is fine)
        ckpt_dst = Path(KAGGLE_ROOT) / "checkpoints"
        ckpt_dst.mkdir(parents=True, exist_ok=True)
        result = subprocess.run(["rclone", "sync",
                                 "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
                                 str(ckpt_dst), "--progress"])
        if result.returncode != 0:
            print("Warning: checkpoint sync returned non-zero - continuing (may be first run).")

        # Pull dataset from Drive (skip if already extracted this session).
        # Uses DATASET_NAME from cell 1 -- Kaggle does its own rclone-based
        # sync here rather than remote_train.py's sync_dataset(), so this is
        # the one place that must independently stay in sync with the name.
        ds_dir = Path(KAGGLE_ROOT) / DATASET_NAME
        if not ds_dir.exists():
            ds_zip = Path(KAGGLE_ROOT) / f"{DATASET_NAME}.zip"
            subprocess.run(["rclone", "copy",
                            f"gdrive:Colab Notebooks/TL-Bot/{DATASET_NAME}.zip",
                            str(Path(KAGGLE_ROOT)), "--progress"], check=True)
            with _zipfile.ZipFile(ds_zip, "r") as zf:
                zf.extractall(Path(KAGGLE_ROOT))
            ds_zip.unlink()

        REPO_DIR = "/kaggle/working/Discord-TL_Bot"
        _clone_or_pull(REPO_DIR)

        # Symlink dataset into repo tree (Models/Datasets/ is gitignored, won't exist after clone).
        _ds_dst = f"{REPO_DIR}/Models/Datasets/{DATASET_NAME}"
        if not os.path.exists(_ds_dst):
            os.makedirs(f"{REPO_DIR}/Models/Datasets", exist_ok=True)
            os.symlink(str(ds_dir), _ds_dst)

        STORAGE_ROOT = KAGGLE_ROOT
        storage_args = ["--storage-root", KAGGLE_ROOT, "--skip-dataset",
                        "--repo-dir", REPO_DIR,
                        "--sync-to", "gdrive:Colab Notebooks/TL-Bot/checkpoints/"]

    case "lightning":
        LIGHTNING_ROOT = "/teamspace/studios/this_studio/TL-Bot"

        os.makedirs(LIGHTNING_ROOT, exist_ok=True)
        REPO_DIR = f"{LIGHTNING_ROOT}/Discord-TL_Bot"
        _clone_or_pull(REPO_DIR)

        STORAGE_ROOT = LIGHTNING_ROOT
        storage_args = ["--storage-root", LIGHTNING_ROOT]

    case "local":
        LOCAL_REPO = ""  # e.g. r"C:\Users\you\Documents\Discord-TL_Bot"
        LOCAL_ROOT = str(Path.home() / "tl-bot-checkpoints")

        Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
        REPO_DIR = LOCAL_REPO or os.getcwd()

        STORAGE_ROOT = LOCAL_ROOT
        storage_args = ["--storage-root", LOCAL_ROOT, "--skip-dataset", "--repo-dir", REPO_DIR]

    case _:
        raise ValueError(f"Unknown PLATFORM: {PLATFORM!r}")

# Unified status summary -- identical shape regardless of platform, printed
# once the platform-specific work above finishes.
print(f"Platform: {PLATFORM}")
print(f"Repo: {REPO_DIR}")
print(f"Checkpoints: {STORAGE_ROOT}/checkpoints")

---
## Cell 3 - Train
Starts or resumes training using the config cell 2 already resolved. Platform-agnostic -- no `PLATFORM` references here.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path as _Path

# Checkpoint dir, mirroring train.py's scoping. Resolved here (not per-platform --
# REPO_DIR/STORAGE_ROOT/storage_args all came from cell 2's single dispatch) so
# cell 4 can reuse it instead of repeating the rule.
_ALL = {"latin", "kana", "hangul", "cjk"}
_sel = _ALL if "all" in SCRIPTS else set(SCRIPTS)
if _sel >= _ALL:
    CKPT_DIR = _Path(STORAGE_ROOT) / "checkpoints"
elif len(SCRIPTS) == 1:
    CKPT_DIR = _Path(STORAGE_ROOT) / "checkpoints" / SCRIPTS[0]
else:
    CKPT_DIR = _Path(STORAGE_ROOT) / "checkpoints" / "_".join(sorted(_sel))
# Mirrors remote_train.py's _make_ckpt_dir(): a non-default DATASET_NAME gets
# its own checkpoint dir so it can never land in and overwrite a
# default-dataset run's checkpoint.
if DATASET_NAME != "char-dataset":
    _suffix = DATASET_NAME[len("char-dataset"):].lstrip("-_") or DATASET_NAME
    CKPT_DIR = CKPT_DIR.parent / f"{CKPT_DIR.name}_{_suffix}"

# Print last training progress from progress.json before launching -- DO NOT REMOVE
# Wrapped: a display problem must never stop the training launch.
_prog = CKPT_DIR / "progress.json"
try:
    if _prog.exists():
        print("[progress]")
        for k, v in json.loads(_prog.read_text()).items():
            if not isinstance(v, (list, dict)):
                print(f"  {k}: {v}")
    else:
        print("[progress] No prior run found - starting fresh.")
except Exception as e:
    print(f"[progress] Could not read progress.json: {e}")

_cmd = [
    "python", "-u", f"{REPO_DIR}/Models/remote_train.py",
    "--skip-clone",
    "--scripts", *SCRIPTS,
    "--epochs", str(EPOCHS),
    "--scheduler", SCHEDULER,
    "--lr", str(LR),
    "--dataset-name", DATASET_NAME,
    *storage_args,
]
if RESUME:
    _cmd.append("--resume")

# Run the child process with its output streamed into the notebook.
#
# subprocess.run() without a pipe is useless here: IPython replaces sys.stdout at
# the Python level only, so a child inherits the kernel's real fd 1 and writes to
# Colab's server log, not this cell. Every setup message and traceback from
# remote_train.py was being discarded that way, leaving a bare CalledProcessError
# with no cause attached. Pipe it and re-print through sys.stdout instead.
print("$ " + " ".join(str(c) for c in _cmd) + "\n", flush=True)
_proc = subprocess.Popen(_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, errors="replace")
_tail = []
for _line in _proc.stdout:
    sys.stdout.write(_line)
    sys.stdout.flush()
    _tail.append(_line)
    del _tail[:-40]
_rc = _proc.wait()
if _rc != 0:
    raise RuntimeError(
        f"remote_train.py exited {_rc}. Last {len(_tail)} lines:\n" + "".join(_tail))

---
## Cell 4 - Final Results
Run once cell 3 finishes. Prints the run summary and the last epoch's metrics from `progress.json`, plus `curves.png`.

Says so and stops if the run has not reached its last epoch. Fields are read from the JSON as they come, so metrics added to `train.py` show up without editing this cell.

The test-set report (per-class precision/recall, top-1/3/5, confused pairs) is printed at the end of cell 3 and is not saved to disk.

In [ ]:
import json

# Final results, read from progress.json in CKPT_DIR (resolved in cell 3).
# Keys come from the file, so metrics added to train.py appear without edits here.

_d = json.loads((CKPT_DIR / "progress.json").read_text())
_hist = _d.get("history", [])

if _d.get("completed", 0) < _d.get("total_epochs", 0):
    print(f"[results] Training unfinished: epoch {_d.get('completed')} of "
          f"{_d.get('total_epochs')}. Re-run once cell 3 completes.")
else:
    print("=" * 72)
    print(f" FINAL RESULTS   {CKPT_DIR}")
    print("=" * 72)
    for k, v in _d.items():
        if not isinstance(v, (list, dict)):
            print(f"  {k:<20}: {v}")

    if _hist:
        print(f"\n  --- last epoch ({_hist[-1].get('epoch')}) ---")
        for k, v in _hist[-1].items():
            print(f"  {k:<20}: {v}")

    if (CKPT_DIR / "curves.png").exists():
        from IPython.display import Image, display
        display(Image(filename=str(CKPT_DIR / "curves.png")))